# Using OmniSVG for Text-to-SVG Generation

This notebook demonstrates how to use the trained OmniSVG model to generate SVGs from text descriptions.

In [ ]:
import os
import sys
import json
import torch
import matplotlib.pyplot as plt
from PIL import Image
import io
import cairosvg

# Add parent directory to path to import OmniSVG modules
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.inference import OmniSVGGenerator
from src.utils import visualize_svg

## Load the Model

First, let's load the trained OmniSVG model. You need to specify the path to the model directory.

In [ ]:
# Path to the model directory
model_dir = "../models/omnisvg/final_model"

# Check if the model directory exists
if not os.path.exists(model_dir):
    print(f"Model directory {model_dir} does not exist. Using a sample model for demonstration.")
    model_dir = "../models/sample_model"  # For demonstration purposes

# Initialize the generator
generator = OmniSVGGenerator(model_dir)

## Generate SVGs from Text

Now let's generate some SVGs from text descriptions:

In [ ]:
def display_generated_svg(svg_content):
    """Display an SVG in the notebook."""
    try:
        # Convert SVG to PNG for display
        png_data = cairosvg.svg2png(bytestring=svg_content.encode('utf-8'))
        img = Image.open(io.BytesIO(png_data))
        
        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
        
        # Print truncated SVG content
        print("SVG Content (truncated):")
        print(svg_content[:300] + '...' if len(svg_content) > 300 else svg_content)
    except Exception as e:
        print(f"Error displaying SVG: {e}")
        print("Raw SVG content:")
        print(svg_content)

In [ ]:
# Example prompts to try
prompts = [
    "A red heart icon",
    "A blue star with yellow outline",
    "A smiling sun with sunglasses",
    "A simple house icon with a chimney",
    "A cartoon cat face"
]

# Choose a prompt
prompt = prompts[0]  # Change the index to try different prompts
print(f"Generating SVG for prompt: '{prompt}'")

# Generate SVG
generated_svgs = generator.generate(
    text=prompt,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    num_samples=1
)

# Display the generated SVG
display_generated_svg(generated_svgs[0])

## Generate Multiple Variations

We can generate multiple variations of the same prompt by adjusting the generation parameters and requesting multiple samples:

In [ ]:
# Choose a prompt
prompt = "A simple tree icon"
print(f"Generating multiple variations for prompt: '{prompt}'")

# Generate multiple SVGs
generated_svgs = generator.generate(
    text=prompt,
    temperature=0.9,  # Higher temperature for more diversity
    top_k=50,
    top_p=0.95,
    num_samples=3
)

# Display all generated SVGs
for i, svg in enumerate(generated_svgs):
    print(f"\nVariation {i+1}:")
    display_generated_svg(svg)

## Experiment with Generation Parameters

Let's experiment with different generation parameters to see how they affect the output:

In [ ]:
def generate_with_params(prompt, temperature, top_k, top_p):
    """Generate an SVG with specific parameters."""
    print(f"Generating with temperature={temperature}, top_k={top_k}, top_p={top_p}")
    
    generated_svgs = generator.generate(
        text=prompt,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        num_samples=1
    )
    
    display_generated_svg(generated_svgs[0])

In [ ]:
# Choose a prompt
prompt = "A mountain landscape icon"

# Low temperature (more deterministic)
generate_with_params(prompt, temperature=0.3, top_k=50, top_p=0.95)

# High temperature (more creative)
generate_with_params(prompt, temperature=1.2, top_k=50, top_p=0.95)

# Low top_k (less variety)
generate_with_params(prompt, temperature=0.7, top_k=10, top_p=0.95)

# Low top_p (more focused)
generate_with_params(prompt, temperature=0.7, top_k=50, top_p=0.5)

## Save Generated SVGs

Let's save some of the generated SVGs to files:

In [ ]:
# Create output directory
output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)

# Choose a prompt
prompt = "A compass navigation icon"
print(f"Generating and saving SVG for prompt: '{prompt}'")

# Generate and save SVGs
file_paths = generator.generate_and_save(
    text=prompt,
    output_dir=output_dir,
    prefix="compass",
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    num_samples=1
)

print(f"Saved SVG to: {file_paths[0]}")
display_generated_svg(open(file_paths[0], 'r').read())

## Batch Generation

We can also generate SVGs for multiple prompts at once:

In [ ]:
# Batch of prompts
batch_prompts = [
    "A simple clock icon",
    "A lightning bolt icon",
    "A cloud with rain"
]

# Generate SVGs for all prompts
batch_results = generator.batch_generate(
    prompts=batch_prompts,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    num_samples=1
)

# Display all generated SVGs
for i, (prompt, svgs) in enumerate(zip(batch_prompts, batch_results)):
    print(f"\nPrompt {i+1}: '{prompt}'")
    display_generated_svg(svgs[0])

## More Complex Examples

Let's try some more complex and detailed prompts:

In [ ]:
complex_prompts = [
    "A cartoon panda character wearing sunglasses and a red cap",
    "A detailed compass with north, south, east, and west indicators and a decorative border",
    "A stylized robot with square head, antenna, and mechanical arms",
    "A mountain landscape with pine trees, a lake, and a small cabin"
]

# Choose a complex prompt
complex_prompt = complex_prompts[0]  # Change the index to try different prompts
print(f"Generating SVG for complex prompt: '{complex_prompt}'")

# Generate SVG with higher max_length for complex SVGs
generated_svgs = generator.generate(
    text=complex_prompt,
    max_length=2048,  # Longer sequence for complex SVGs
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    num_samples=1
)

# Display the generated SVG
display_generated_svg(generated_svgs[0])

## Conclusion

In this notebook, we've demonstrated how to use the OmniSVG model to generate SVGs from text descriptions. We've explored different generation parameters and shown how to create multiple variations of the same prompt.

The model can generate a wide range of SVGs, from simple icons to more complex illustrations, making it a powerful tool for designers and creative professionals.